In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG = "rio_dataengineering"
AUDIT_SCHEMA = "audit"
PIPELINE_NAME = "people-analytics-lakehouse-pipeline"

RUN_TIMESTAMP = datetime.now()

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {AUDIT_SCHEMA}")

print("Pipeline audit configuration ready.")

In [0]:
quality_history = spark.table(
    f"{CATALOG}.{AUDIT_SCHEMA}.data_quality_results"
)

latest_check_timestamp = (
    quality_history
    .agg(F.max("checked_at").alias("latest_checked_at"))
    .first()["latest_checked_at"]
)

latest_quality_results = (
    quality_history
    .filter(F.col("checked_at") == F.lit(latest_check_timestamp))
)

print(f"Latest quality batch: {latest_check_timestamp}")
print(f"Checks found: {latest_quality_results.count()}")

In [0]:
quality_metrics = (
    latest_quality_results
    .agg(
        F.count("*").alias("total_checks"),
        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).alias("passed_checks"),
        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).alias("failed_checks"),
        F.sum("failure_count").alias("total_failed_records")
    )
    .first()
)

total_checks = int(quality_metrics["total_checks"])
passed_checks = int(quality_metrics["passed_checks"])
failed_checks = int(quality_metrics["failed_checks"])
total_failed_records = int(
    quality_metrics["total_failed_records"] or 0
)

run_status = "SUCCESS" if failed_checks == 0 else "FAILED"

print(f"Pipeline status: {run_status}")
print(f"Passed checks: {passed_checks}/{total_checks}")

In [0]:
bronze_rows = sum(
    spark.table(f"{CATALOG}.bronze.{table_name}").count()
    for table_name in [
        "employees",
        "departments",
        "locations",
        "payroll",
        "training",
        "engagement",
        "leave",
    ]
)

silver_rows = sum(
    spark.table(f"{CATALOG}.silver.{table_name}").count()
    for table_name in [
        "employees",
        "departments",
        "locations",
        "payroll",
        "training",
        "engagement",
        "leave",
    ]
)

gold_rows = sum(
    spark.table(f"{CATALOG}.gold.{table_name}").count()
    for table_name in [
        "workforce_overview",
        "department_kpis",
        "monthly_payroll_summary",
        "location_kpis",
        "employee_analytics",
        "training_summary",
        "engagement_summary",
        "leave_summary",
    ]
)

print(f"Bronze rows: {bronze_rows}")
print(f"Silver rows: {silver_rows}")
print(f"Gold rows: {gold_rows}")

In [0]:
run_schema = StructType(
    [
        StructField("pipeline_name", StringType(), False),
        StructField("run_timestamp", TimestampType(), False),
        StructField("quality_checked_at", TimestampType(), False),
        StructField("run_status", StringType(), False),
        StructField("total_checks", LongType(), False),
        StructField("passed_checks", LongType(), False),
        StructField("failed_checks", LongType(), False),
        StructField("total_failed_records", LongType(), False),
        StructField("bronze_row_count", LongType(), False),
        StructField("silver_row_count", LongType(), False),
        StructField("gold_row_count", LongType(), False),
    ]
)

run_record = [
    (
        PIPELINE_NAME,
        RUN_TIMESTAMP,
        latest_check_timestamp,
        run_status,
        total_checks,
        passed_checks,
        failed_checks,
        total_failed_records,
        bronze_rows,
        silver_rows,
        gold_rows,
    )
]

pipeline_run_df = spark.createDataFrame(
    run_record,
    schema=run_schema,
)

display(pipeline_run_df)

In [0]:
(
    pipeline_run_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{CATALOG}.{AUDIT_SCHEMA}.pipeline_run_history"
    )
)

print("Pipeline run audit record saved.")

In [0]:
audit_history = spark.table(
    f"{CATALOG}.{AUDIT_SCHEMA}.pipeline_run_history"
)

display(
    audit_history.orderBy(
        F.col("run_timestamp").desc()
    )
)

In [0]:
if run_status != "SUCCESS":
    raise ValueError(
        f"Pipeline audit recorded an unsuccessful run: {run_status}"
    )

print("Pipeline monitoring and audit completed successfully.")